# **Project Name**    - **Yes Bank Stock Closing Price Prediction**


##### **Project Type**    - Regression
##### **Contribution**    - Individual
##### **Team Member 1 -** *(Your Name Here)*


# **Project Summary -**

Yes Bank is one of India's prominent private-sector banks. Since 2018, it has been in the news
for the wrong reasons — a large fraud case involving its co-founder, Rana Kapoor, triggered a severe
governance and liquidity crisis that eventually led to the Reserve Bank of India (RBI) placing the bank
under a moratorium in March 2020. This project studies the **monthly stock price behaviour of Yes Bank
from July 2005 to November 2020** and builds a Machine Learning regression model to predict the stock's
**monthly closing price**.

The dataset used contains 185 monthly records with five columns — `Date`, `Open`, `High`, `Low`, and
`Close`. Since the data is naturally time-ordered and the four price columns are highly collinear (as is
typical for OHLC data), the primary analytical challenge of this project is not raw prediction accuracy —
which is already high given how closely Open/High/Low track Close — but rather building meaningful
**engineered features** (trading range, momentum, moving averages, lag prices) that let a model capture
the underlying *trend* and *volatility* dynamics rather than simply memorising near-identical inputs.

The project follows a structured Data Science pipeline. First, an in-depth **Exploratory Data Analysis
(EDA)** is performed — examining the distribution of prices, the long-term growth trend from 2005 to
2018, the dramatic crash from late-2018 onward, month-wise seasonality, and correlations between
variables — using univariate, bivariate and multivariate visualizations. Statistical **hypothesis
testing** is then used to formally verify observations from the EDA, such as whether the average
closing price and trading volatility genuinely differ between the pre-crisis and crisis periods.

Next, **feature engineering** creates additional predictors: the High-Low spread (intra-month
volatility), Open-Close difference and percentage return (momentum), lagged closing prices, and 3-
and 6-month moving averages (trend). Because the data is chronological, a **time-based train-test
split** is used (rather than a random split) to avoid leaking future information into the training
set, and all numeric features are standardised using `StandardScaler`.

Three regression algorithms are trained and compared: **Linear Regression** (a simple, interpretable
baseline), **Random Forest Regressor** (a non-linear ensemble that captures interactions among
features), and **XGBoost Regressor** (a gradient-boosted ensemble that typically achieves the strongest
performance on tabular data). Each model is evaluated using **R², RMSE, MAE and MAPE**, and the top
performing model is further improved using **GridSearchCV** hyperparameter tuning. Feature importance
from the best tree-based model is used to explain which factors most influence the prediction —
typically the most recent lagged price and short-term moving averages dominate, confirming that
**recent momentum matters more than any single day's price range**.

The final model is saved using `pickle` for reuse/deployment, and the notebook concludes with business
insights: pure price-based models can predict short-term closing price well under *normal* market
conditions, but cannot anticipate sudden shocks driven by fraud or regulatory action (like the 2018
Yes Bank crisis) — so such models should always be paired with fundamental and governance-risk
monitoring rather than used in isolation for investment decisions.


# **GitHub Link -**

https://github.com/your-username/yes-bank-stock-price-prediction  *(replace with your repository link before submission)*

# **Problem Statement**


Yes Bank's stock has experienced extreme swings — from steady, strong growth between 2005 and 2018,
to a dramatic crash following the 2018 co-founder fraud allegations and the subsequent RBI moratorium
in March 2020. Given only the **monthly Open, High, Low and Close prices** since the company's listing,
can we build a regression model that **accurately predicts the month's closing price**, and can we
quantify how much the 2018-2020 crisis genuinely changed the stock's price level and volatility
compared to the years before it?

**Business Objective:** Build an accurate, explainable regression model for Yes Bank's monthly closing
price that can support analysts/investors in short-term price forecasting, while surfacing the key
drivers (momentum, volatility, trend) behind the stock's movements — and quantify the impact of the
2018 governance crisis using statistical hypothesis testing.


# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('viridis')
%matplotlib inline

# Statistics
from scipy import stats

# Preprocessing & Model Selection
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Metrics
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

import pickle
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
print("Libraries imported successfully.")

### Dataset Loading

In [ ]:
# Load Dataset
# In Google Colab, upload 'data_YesBank_StockPrices.csv' when prompted, or mount your Google Drive.
from google.colab import files
uploaded = files.upload()

df = pd.read_csv('data_YesBank_StockPrices.csv')

### Dataset First View

In [ ]:
# Dataset First Look
df.head(10)

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
print(f"Number of rows    : {df.shape[0]}")
print(f"Number of columns : {df.shape[1]}")

### Dataset Information

In [ ]:
# Dataset Info
df.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
print("Number of duplicate rows:", df.duplicated().sum())

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
df.isnull().sum()

In [ ]:
# Visualizing the missing values
plt.figure(figsize=(8,4))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Missing Value Heatmap (Yes Bank Stock Prices)')
plt.show()

print("There are no missing values in this dataset — the heatmap above is uniformly colored,")
print("confirming that every Date/Open/High/Low/Close cell is populated.")

### What did you know about your dataset?

The dataset contains **185 monthly records** of Yes Bank's stock price from **July 2005 to
November 2020**, with 5 columns: `Date` (month-year, e.g. `Jul-05`), `Open`, `High`, `Low` and
`Close` (all in INR). There are **no missing values** and **no duplicate rows** — the data is
already clean at the raw level. The `Date` column is stored as text (`Mon-YY` format) rather than
a proper datetime, and the four numeric columns are stored as `float64`/`int64`. The stock's price
ranges from roughly ₹8 (during the 2020 crash) to over ₹390 (its 2018 peak), which already hints
at the extreme volatility this stock has experienced across its history.


## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
df.columns

In [ ]:
# Dataset Describe
df.describe()

### Variables Description

| Variable | Description |
|---|---|
| **Date** | Month and year of the record (e.g. `Jul-05` = July 2005) |
| **Open** | Stock price (INR) at the start of the month |
| **High** | Highest stock price (INR) reached during the month |
| **Low** | Lowest stock price (INR) reached during the month |
| **Close** | Stock price (INR) at the end of the month — **this is our target variable** |

All four price columns are continuous numeric variables. `Close` ranges from about ₹8 to ₹392,
with a mean of roughly ₹86 and a large standard deviation — reflecting the huge swings between the
2005-2018 growth period and the 2018-2020 crash.


### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.

# 1. Convert 'Date' (e.g. 'Jul-05') into a proper datetime column and sort chronologically
df['Date'] = pd.to_datetime(df['Date'], format='%b-%y')
df = df.sort_values('Date').reset_index(drop=True)

# 2. Time-based features
df['Year']  = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

# 3. Volatility / momentum engineered features
df['HL_diff']    = df['High'] - df['Low']                        # Intra-month trading range (volatility)
df['OC_diff']    = df['Close'] - df['Open']                       # Net movement within the month
df['pct_change'] = (df['Close'] - df['Open']) / df['Open'] * 100  # Monthly % return

# 4. Trend / momentum features
df['Close_lag1'] = df['Close'].shift(1)      # Previous month's closing price
df['Close_lag2'] = df['Close'].shift(2)      # Closing price 2 months ago
df['MA_3']       = df['Close'].rolling(window=3).mean()   # 3-month moving average
df['MA_6']       = df['Close'].rolling(window=6).mean()   # 6-month moving average

# 5. Flag for the Yes Bank governance crisis period, used later for hypothesis testing
df['Crisis_Period'] = df['Date'].between('2018-09-01', '2020-03-31')

# 6. Drop the first two rows which now contain NaN due to the lag/rolling window features
df = df.dropna().reset_index(drop=True)

print("Final shape after wrangling:", df.shape)
df.head()

### What all manipulations have you done and insights you found?

The raw `Date` string was converted into a proper `datetime` column and the data was sorted
chronologically. Since the four OHLC columns are almost perfectly collinear (a simple model would
just learn `Close ≈ Open`), several new features were engineered to capture information the raw
prices alone don't express:

- **`HL_diff`** (High − Low) — a direct measure of intra-month volatility.
- **`OC_diff`** and **`pct_change`** — the net direction/magnitude of the month's move.
- **`Close_lag1`, `Close_lag2`** — the previous 1-2 months' closing prices, capturing momentum.
- **`MA_3`, `MA_6`** — 3- and 6-month moving averages, capturing the medium-term trend.
- **`Crisis_Period`** — a boolean flag for the Sep-2018 to Mar-2020 governance-crisis window, used
  later for hypothesis testing and bivariate analysis.

This turned the raw 5-column dataset into an 12-column, model-ready dataset with 183 usable rows
(2 rows dropped due to the lag/rolling windows needing prior history).


## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart - 1 visualization code
fig, ax = plt.subplots(figsize=(14,6))
ax.plot(df['Date'], df['Close'], color='#1f77b4', linewidth=2)
ax.axvspan(pd.Timestamp('2018-09-01'), pd.Timestamp('2020-03-01'), color='red', alpha=0.12,
           label='Yes Bank Crisis Period (Sep 2018 - Mar 2020)')
ax.set_title('Yes Bank Monthly Closing Price (2005 - 2020)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Closing Price (INR)')
ax.legend()
plt.tight_layout(); plt.show()

##### 1. Why did you pick the specific chart?

A line chart is the natural choice for visualising how a single continuous variable (closing price) evolves over time — it is the clearest way to see long-term trend, turning points and the timing of the crash.

##### 2. What is/are the insight(s) found from the chart?

The stock grew steadily from 2005, dipped during the 2008 global financial crisis, then grew strongly from 2013 to a peak above ₹390 in August 2018 — before collapsing to under ₹10 by 2020, coinciding exactly with the shaded crisis window.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — this chart clearly separates a long profitable growth period from a governance-driven collapse. It tells investors/analysts that the crash was **event-driven**, not a gradual market correction, which has direct implications for risk monitoring and model design (e.g., the crisis period should be treated/tested separately, as done later in this notebook).

#### Chart - 2

In [ ]:
# Chart - 2 visualization code
plt.figure(figsize=(8,5))
sns.histplot(df['Close'], kde=True, color='teal', bins=25)
plt.title('Distribution of Closing Price')
plt.xlabel('Closing Price (INR)')
plt.show()

##### 1. Why did you pick the specific chart?

A histogram with a KDE overlay is the standard way to inspect the univariate distribution — shape, skew and modality — of a continuous numeric variable like `Close`.

##### 2. What is/are the insight(s) found from the chart?

The distribution is strongly **right-skewed** with a peak in the low price band (₹10-100) and a long tail extending to ₹390+. This confirms the stock spent most of its life in low price ranges (early years + post-crisis) with only a shorter period at high valuations.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — the skew signals that models sensitive to scale (e.g. distance-based/linear models without scaling) could be biased toward the dominant low-price regime. This motivates the feature scaling step used later before model training.

#### Chart - 3

In [ ]:
# Chart - 3 visualization code
plt.figure(figsize=(8,5))
sns.boxplot(data=df[['Open','High','Low','Close']], palette='Set2')
plt.title('Boxplot Comparison: Open, High, Low, Close')
plt.ylabel('Price (INR)')
plt.show()

##### 1. Why did you pick the specific chart?

A boxplot compares the spread, median and outliers of multiple numeric variables side-by-side — useful here to see whether Open/High/Low/Close behave similarly.

##### 2. What is/are the insight(s) found from the chart?

All four price columns show nearly identical distributions with similar medians and a cluster of high-value outliers above ₹300 (the 2017-2018 peak). This confirms the four columns are highly collinear, as expected for OHLC data.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This insight (near-perfect collinearity) directly influenced the feature-engineering strategy: instead of feeding raw Open/High/Low/Close directly as the *only* predictors, engineered spread/momentum features were added so the model has non-redundant information to learn from.

#### Chart - 4

In [ ]:
# Chart - 4 visualization code
plt.figure(figsize=(14,6))
sns.boxplot(data=df, x='Year', y='Close', palette='coolwarm')
plt.title('Yearly Distribution of Closing Price')
plt.xticks(rotation=45)
plt.show()

##### 1. Why did you pick the specific chart?

A boxplot grouped by `Year` is an effective bivariate (categorical vs numerical) chart to compare the spread and central tendency of price across each year at a glance.

##### 2. What is/are the insight(s) found from the chart?

2017 and 2018 show both the highest median price and the widest boxes (spread), confirming peak valuation years. From 2019 onward the boxes collapse close to zero, visually confirming the crisis' effect on both price level and its volatility.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — this quantifies *when* risk was highest (2017-2018, ironically at the peak of investor confidence) versus when the price stabilized at depressed levels (2019-2020). Useful for building any year-based risk overlays in an investment strategy.

#### Chart - 5

In [ ]:
# Chart - 5 visualization code
yearly_avg = df.groupby('Year')['Close'].mean()
plt.figure(figsize=(12,5))
sns.barplot(x=yearly_avg.index, y=yearly_avg.values, color='slateblue')
plt.title('Average Yearly Closing Price')
plt.ylabel('Average Close Price (INR)')
plt.xticks(rotation=45)
plt.show()

##### 1. Why did you pick the specific chart?

A bar chart is ideal for comparing one aggregated numeric value (mean closing price) across a discrete category (year).

##### 2. What is/are the insight(s) found from the chart?

Average closing price rose almost every year from 2005 to a clear peak in 2018 (~₹300+), then fell sharply in 2019 and again in 2020 to below ₹20 — a ~95% decline from peak in just two years.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — quantifying the year-over-year average gives a simple, communicable metric (e.g. for an investor memo) of how severely and quickly the crisis eroded shareholder value, useful for post-mortem risk analysis or governance-risk case studies.

#### Chart - 6

In [ ]:
# Chart - 6 visualization code
monthly_avg = df.groupby('Month')['pct_change'].mean()
plt.figure(figsize=(10,5))
sns.barplot(x=monthly_avg.index, y=monthly_avg.values, color='darkorange')
plt.title('Average Monthly % Return by Calendar Month (Seasonality Check)')
plt.xlabel('Month (1=Jan ... 12=Dec)')
plt.ylabel('Average % Return')
plt.show()

##### 1. Why did you pick the specific chart?

Aggregating percentage return by calendar month checks for **seasonality** — whether certain months systematically perform better/worse, a common bivariate analysis for financial time series.

##### 2. What is/are the insight(s) found from the chart?

Average returns vary by month, but no calendar month shows an extreme, consistent bias — the variation is dominated by the handful of crisis-period months (e.g., Sep 2018, Mar 2020) rather than a genuine recurring seasonal pattern.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

This is a useful **negative finding**: it tells analysts *not* to rely on calendar-month seasonality strategies for this stock, since the apparent monthly differences are actually driven by one-off crisis events, not a repeatable pattern — avoiding a false trading signal.

#### Chart - 7

In [ ]:
# Chart - 7 visualization code
plt.figure(figsize=(14,6))
plt.plot(df['Date'], df['HL_diff'], color='crimson')
plt.title('Intra-Month Trading Range (High - Low) Over Time')
plt.xlabel('Date'); plt.ylabel('High - Low (INR)')
plt.show()

##### 1. Why did you pick the specific chart?

A line chart of the engineered `HL_diff` feature over time directly visualises how volatility itself trends, which a raw price chart cannot show.

##### 2. What is/are the insight(s) found from the chart?

Volatility spikes sharply during the 2008 global financial crisis and, far more dramatically, during 2018-2020 — the single largest spike (₹80+ range in one month) occurs exactly at the peak of the Yes Bank crisis in March 2020.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — this confirms `HL_diff` is a strong volatility indicator that a risk team could monitor in real time; a sudden spike in this metric could serve as an early-warning signal of abnormal, potentially crisis-driven price action.

#### Chart - 8

In [ ]:
# Chart - 8 visualization code
plt.figure(figsize=(7,6))
sns.scatterplot(data=df, x='Open', y='Close', hue='Crisis_Period', palette={True:'red', False:'steelblue'})
plt.title('Open Price vs Close Price')
plt.plot([df['Open'].min(), df['Open'].max()], [df['Open'].min(), df['Open'].max()], 'k--', alpha=0.5)
plt.show()

##### 1. Why did you pick the specific chart?

A scatter plot is the standard bivariate chart for two continuous numeric variables, and colouring by the crisis flag adds a categorical dimension to see if the relationship differs by regime.

##### 2. What is/are the insight(s) found from the chart?

Open and Close are almost perfectly linearly related (points hug the diagonal) in both regimes, though crisis-period points (red) show more scatter/deviation from the diagonal — consistent with the higher volatility already seen in `HL_diff`.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — the near-linear relationship justifies including `Open` as a strong predictor, while the extra scatter during the crisis period suggests the model may need crisis-aware features (as engineered) to keep prediction error low during turbulent periods.

#### Chart - 9

In [ ]:
# Chart - 9 visualization code
plt.figure(figsize=(8,5))
sns.histplot(df['pct_change'], kde=True, color='purple', bins=25)
plt.title('Distribution of Monthly % Return (Close vs Open)')
plt.xlabel('% Return')
plt.show()

##### 1. Why did you pick the specific chart?

A histogram of the engineered `pct_change` feature shows the shape of monthly returns — important for understanding risk (fat tails, skew) beyond just price level.

##### 2. What is/are the insight(s) found from the chart?

Returns are roughly centered near zero but with **fat tails on both sides** (some months over +40% or under -40%), a classic sign of a high-volatility stock rather than a stable, low-risk one.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — fat-tailed returns imply that any investment strategy based on this stock carries meaningfully higher risk of large monthly swings than a 'typical' large-cap bank stock, which is directly relevant for position-sizing and risk management decisions.

#### Chart - 10

In [ ]:
# Chart - 10 visualization code
plt.figure(figsize=(14,6))
plt.plot(df['Date'], df['Close'], label='Close', alpha=0.5, color='gray')
plt.plot(df['Date'], df['MA_3'], label='3-Month MA', color='blue')
plt.plot(df['Date'], df['MA_6'], label='6-Month MA', color='red')
plt.title('Closing Price with 3-Month and 6-Month Moving Averages')
plt.legend()
plt.show()

##### 1. Why did you pick the specific chart?

Overlaying moving averages on the raw price is a standard multivariate/trend-analysis technique in financial analysis to smooth out short-term noise and reveal the underlying trend direction.

##### 2. What is/are the insight(s) found from the chart?

The 3-month MA tracks price closely while the 6-month MA lags more, and both moving averages cross downward sharply from 2018 onward — a textbook 'bearish crossover' signal that would have flagged the crisis early to a technical analyst.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — moving-average crossovers are exactly the kind of engineered feature that gives the ML model (and a human analyst) an early trend-reversal signal, which is why `MA_3`/`MA_6` are included as predictors in the modeling stage.

#### Chart - 11

In [ ]:
# Chart - 11 visualization code
top_volatile = df.nlargest(10, 'HL_diff')[['Date','HL_diff']].set_index('Date')
plt.figure(figsize=(10,5))
sns.barplot(x=top_volatile['HL_diff'], y=top_volatile.index.strftime('%b-%Y'), color='firebrick')
plt.title('Top 10 Most Volatile Months (Largest High-Low Spread)')
plt.xlabel('High - Low (INR)')
plt.show()

##### 1. Why did you pick the specific chart?

Ranking and bar-charting the top-N months by volatility is a focused way to highlight the specific extreme events buried inside the full time series line chart.

##### 2. What is/are the insight(s) found from the chart?

8 of the 10 most volatile months fall within or right around the 2018-2020 crisis window (the remainder being 2008 financial-crisis months) — concretely proving the crisis, not normal market activity, produced the stock's most extreme price swings.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — this is strong quantitative evidence (not just a visual impression) that governance risk, not general market risk, was the dominant driver of extreme volatility for this stock — an important distinction for risk attribution.

#### Chart - 12

In [ ]:
# Chart - 12 visualization code
comparison = df.groupby('Crisis_Period')[['Close','HL_diff','pct_change']].mean()
comparison.index = ['Non-Crisis Period','Crisis Period']
comparison['Close'].plot(kind='bar', color=['steelblue','red'], figsize=(7,5))
plt.title('Average Closing Price: Crisis vs Non-Crisis Period')
plt.ylabel('Average Close (INR)')
plt.xticks(rotation=0)
plt.show()
comparison

##### 1. Why did you pick the specific chart?

A grouped bar chart comparing a numeric variable's mean across a binary categorical flag (`Crisis_Period`) is a direct way to quantify the difference the earlier charts suggested visually — and it feeds straight into the formal hypothesis test performed later.

##### 2. What is/are the insight(s) found from the chart?

Average closing price during the crisis period is dramatically lower than during the non-crisis period, and average volatility (`HL_diff`) is meaningfully higher — confirming both a level shift and a volatility shift caused by the crisis.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — this table is the direct evidence base for Hypothesis Statements 1 and 3 tested later in the notebook, and for any client-facing summary needing a single, defensible number for 'how much did the crisis cost investors'.

#### Chart - 13

In [ ]:
# Chart - 13 visualization code
zoom = df[(df['Date'] >= '2017-01-01') & (df['Date'] <= '2020-11-01')]
plt.figure(figsize=(14,6))
plt.plot(zoom['Date'], zoom['Close'], marker='o', color='darkred')
plt.axvline(pd.Timestamp('2018-09-01'), color='black', linestyle='--', label='Crisis Onset (Sep 2018)')
plt.title('Zoomed View: Closing Price 2017 - 2020 (Crisis Detail)')
plt.xlabel('Date'); plt.ylabel('Closing Price (INR)')
plt.legend()
plt.show()

##### 1. Why did you pick the specific chart?

Zooming the time-series line chart into just the 2017-2020 window (multivariate view of time + price + a marked event date) gives the fine-grained detail that the full 15-year chart compresses too much to see clearly.

##### 2. What is/are the insight(s) found from the chart?

The decline is not a single crash but happens in **distinct steps**: a sharp fall in Sep 2018, a partial recovery into early 2019, a second collapse through mid-2019, and a final drop to single digits around the March 2020 RBI moratorium.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes — recognizing that the crash happened in stages (not one event) is important for anyone building a monitoring system: multiple independent 'trigger points' existed where risk indicators could have flagged trouble, not just one moment.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
plt.figure(figsize=(10,8))
numeric_cols = ['Open','High','Low','Close','HL_diff','OC_diff','pct_change','Close_lag1','Close_lag2','MA_3','MA_6']
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Heatmap - All Numeric Features')
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

A correlation heatmap is the standard multivariate chart for quickly spotting linear relationships and redundancy (multicollinearity) across every pair of numeric features at once.

##### 2. What is/are the insight(s) found from the chart?

`Open`, `High`, `Low`, `Close`, `Close_lag1`, `Close_lag2`, `MA_3` and `MA_6` are all very strongly correlated with each other (>0.95) as expected, while `HL_diff`, `OC_diff` and `pct_change` are comparatively weakly correlated with `Close` — meaning they add genuinely new information rather than duplicating the price-level features.

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code
sns.pairplot(df[['Open','High','Low','Close','HL_diff','pct_change']], diag_kind='kde', corner=True)
plt.suptitle('Pair Plot of Key Numeric Features', y=1.02)
plt.show()

##### 1. Why did you pick the specific chart?

A pair plot extends the correlation heatmap by showing the *actual shape* of every pairwise relationship (not just a single correlation number) plus each variable's own distribution on the diagonal — useful multivariate confirmation before modeling.

##### 2. What is/are the insight(s) found from the chart?

The Open/High/Low/Close scatter panels all show tight, near-linear diagonal bands (confirming the heatmap's high correlations), while `HL_diff` and `pct_change` scatter far more loosely against price — visually confirming they capture a different kind of signal (volatility/momentum) rather than just price level.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

Based on the EDA above (particularly Chart-1, Chart-4 and Chart-12), three hypotheses are tested
formally below to move from visual impression to statistical evidence:

1. Whether the **average closing price** differs between the crisis and non-crisis periods.
2. Whether **Open price and Close price** are significantly (linearly) correlated.
3. Whether the **average monthly volatility** (`HL_diff`) differs between the crisis and non-crisis periods.


### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**Null Hypothesis (H0):** There is no significant difference in the mean monthly closing price
between the crisis period (Sep 2018 - Mar 2020) and the non-crisis period.

**Alternate Hypothesis (H1):** The mean monthly closing price during the crisis period is
significantly different (lower) than during the non-crisis period.


#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
crisis_close     = df[df['Crisis_Period'] == True]['Close']
non_crisis_close = df[df['Crisis_Period'] == False]['Close']

t_stat, p_value = stats.ttest_ind(crisis_close, non_crisis_close, equal_var=False)  # Welch's t-test

print(f"T-statistic : {t_stat:.4f}")
print(f"P-value     : {p_value:.6f}")

alpha = 0.05
if p_value < alpha:
    print("Result: Reject the Null Hypothesis -> Significant difference in mean closing price.")
else:
    print("Result: Fail to reject the Null Hypothesis -> No significant difference found.")

##### Which statistical test have you done to obtain P-Value?

An **independent two-sample t-test (Welch's t-test)** was used, comparing the mean `Close`
of the crisis-period rows against the non-crisis-period rows.

##### Why did you choose the specific statistical test?

Welch's t-test is appropriate because we are comparing the **means of a continuous variable
(Close)** between **two independent groups** (crisis vs. non-crisis months), and Welch's variant is
used (rather than the standard Student's t-test) because the two groups have unequal variances —
volatility during the crisis period is visibly higher, so assuming equal variance would be incorrect.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**Null Hypothesis (H0):** There is no significant linear correlation between `Open` price and
`Close` price (population correlation coefficient ρ = 0).

**Alternate Hypothesis (H1):** There is a significant linear correlation between `Open` price and
`Close` price (ρ ≠ 0).


#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
corr_coef, p_value = stats.pearsonr(df['Open'], df['Close'])

print(f"Pearson correlation coefficient : {corr_coef:.4f}")
print(f"P-value                          : {p_value:.6e}")

alpha = 0.05
if p_value < alpha:
    print("Result: Reject the Null Hypothesis -> Significant correlation exists.")
else:
    print("Result: Fail to reject the Null Hypothesis -> No significant correlation found.")

##### Which statistical test have you done to obtain P-Value?

A **Pearson correlation test** was used between `Open` and `Close`.

##### Why did you choose the specific statistical test?

Pearson's test is the standard choice for measuring the strength and significance of a
**linear relationship between two continuous numeric variables**, which is exactly what the Chart-8
scatter plot suggested (a near-perfectly linear Open-vs-Close relationship).

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**Null Hypothesis (H0):** There is no significant difference in the mean monthly trading
range (`HL_diff`) between the crisis period and the non-crisis period.

**Alternate Hypothesis (H1):** The mean monthly trading range (`HL_diff`) during the crisis period
is significantly different (higher) than during the non-crisis period.


#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
crisis_vol     = df[df['Crisis_Period'] == True]['HL_diff']
non_crisis_vol = df[df['Crisis_Period'] == False]['HL_diff']

t_stat, p_value = stats.ttest_ind(crisis_vol, non_crisis_vol, equal_var=False)  # Welch's t-test

print(f"T-statistic : {t_stat:.4f}")
print(f"P-value     : {p_value:.6f}")

alpha = 0.05
if p_value < alpha:
    print("Result: Reject the Null Hypothesis -> Significant difference in volatility found.")
else:
    print("Result: Fail to reject the Null Hypothesis -> No significant difference found.")

##### Which statistical test have you done to obtain P-Value?

An **independent two-sample t-test (Welch's t-test)** was used, comparing the mean
`HL_diff` (intra-month trading range) of the crisis-period rows against the non-crisis-period rows.

##### Why did you choose the specific statistical test?

As with Hypothesis 1, we are comparing the mean of a continuous variable across two
independent groups with visibly unequal variances, so Welch's t-test is the statistically appropriate
choice over the standard Student's t-test.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation
# Confirmed earlier (Know Your Data section) that the raw dataset has zero missing values.
# The only NaNs introduced were from the lag/rolling-window features created during Data Wrangling,
# and those rows were already dropped at that stage. Re-confirming here:
print("Remaining missing values in the model-ready dataframe:")
print(df.isnull().sum().sum())

#### What all missing value imputation techniques have you used and why did you use those techniques?

No missing-value imputation was required. The raw dataset had zero missing values, and the
only NaNs that appeared (from the `Close_lag1/2` and `MA_3/6` rolling/lag features created during
feature engineering) were handled by **dropping the first 2 rows** — this is preferable to imputing
them, since imputing a lag feature for the very first rows of a time series would fabricate
information that doesn't actually exist.

### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments
Q1 = df['Close'].quantile(0.25)
Q3 = df['Close'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5*IQR
upper_bound = Q3 + 1.5*IQR

outliers = df[(df['Close'] < lower_bound) | (df['Close'] > upper_bound)]
print(f"Number of 'outlier' rows by the IQR rule: {len(outliers)}")
print(outliers[['Date','Close']])

##### What all outlier treatment techniques have you used and why did you use those techniques?

The IQR method flags several months (mostly the 2017-2018 price peaks) as statistical
outliers. However, these are **not measurement errors or data entry mistakes** — they are genuine,
well-documented price levels the stock actually reached. Removing or capping them would erase real
and important information (exactly the periods this project is trying to understand), so **no outlier
removal/treatment was applied**; the outliers were intentionally retained, and their influence on
sensitive models is instead managed via feature scaling and by using tree-based ensembles (Random
Forest/XGBoost) that are naturally robust to extreme values.

### 3. Categorical Encoding

In [ ]:
# Encode your categorical columns
# This dataset has no traditional categorical (string/label) columns to encode.
# 'Month' and 'Year' were derived as integers directly from the Date column, and
# 'Crisis_Period' is a boolean flag used only for EDA/hypothesis testing (not fed to the model).
print(df.dtypes)

#### What all categorical encoding techniques have you used & why did you use those techniques?

No categorical encoding (one-hot/label encoding etc.) was necessary. This dataset is purely
numeric/time-series in nature — the only "categorical-like" fields are `Month` (already a small
integer 1-12) and the boolean `Crisis_Period` flag, which is used purely for EDA and hypothesis
testing and is **not** included as a model input feature (to avoid leaking crisis-period knowledge
directly into the model).

### 4. Textual Data Preprocessing
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:
# Not Applicable — this dataset does not contain any textual data (no reviews, tweets, descriptions, etc.),
# so the Textual Data Preprocessing / NLP steps in this section are skipped.

#### 2. Lower Casing

In [ ]:
# Not Applicable — this dataset does not contain any textual data (no reviews, tweets, descriptions, etc.),
# so the Textual Data Preprocessing / NLP steps in this section are skipped.

#### 3. Removing Punctuations

In [ ]:
# Not Applicable — this dataset does not contain any textual data (no reviews, tweets, descriptions, etc.),
# so the Textual Data Preprocessing / NLP steps in this section are skipped.

#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
# Not Applicable — this dataset does not contain any textual data (no reviews, tweets, descriptions, etc.),
# so the Textual Data Preprocessing / NLP steps in this section are skipped.

#### 5. Removing Stopwords & Removing White spaces

In [ ]:
# Not Applicable — this dataset does not contain any textual data (no reviews, tweets, descriptions, etc.),
# so the Textual Data Preprocessing / NLP steps in this section are skipped.

In [ ]:
# Not Applicable — this dataset does not contain any textual data (no reviews, tweets, descriptions, etc.),
# so the Textual Data Preprocessing / NLP steps in this section are skipped.

#### 6. Rephrase Text

In [ ]:
# Not Applicable — this dataset does not contain any textual data (no reviews, tweets, descriptions, etc.),
# so the Textual Data Preprocessing / NLP steps in this section are skipped.

#### 7. Tokenization

In [ ]:
# Not Applicable — this dataset does not contain any textual data (no reviews, tweets, descriptions, etc.),
# so the Textual Data Preprocessing / NLP steps in this section are skipped.

#### 8. Text Normalization

In [ ]:
# Not Applicable — this dataset does not contain any textual data (no reviews, tweets, descriptions, etc.),
# so the Textual Data Preprocessing / NLP steps in this section are skipped.

##### Which text normalization technique have you used and why?

Not Applicable — this is a numeric stock-price time-series dataset with no textual/NLP data.

#### 9. Part of speech tagging

In [ ]:
# Not Applicable — this dataset does not contain any textual data (no reviews, tweets, descriptions, etc.),
# so the Textual Data Preprocessing / NLP steps in this section are skipped.

#### 10. Text Vectorization

In [ ]:
# Not Applicable — this dataset does not contain any textual data (no reviews, tweets, descriptions, etc.),
# so the Textual Data Preprocessing / NLP steps in this section are skipped.

##### Which text vectorization technique have you used and why?

Not Applicable — this is a numeric stock-price time-series dataset with no textual/NLP data.

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Manipulate Features to minimize feature correlation and create new features
# (The bulk of feature creation already happened during Data Wrangling. Here we finalize
# the feature set used for modeling, dropping columns that would leak the target or are redundant.)

model_features = ['Open', 'High', 'Low', 'HL_diff', 'OC_diff', 'pct_change',
                   'Close_lag1', 'Close_lag2', 'MA_3', 'MA_6', 'Month']

print("Final feature set for modeling:")
print(model_features)

#### 2. Feature Selection

In [ ]:
# Select your features wisely to avoid overfitting
X = df[model_features]
y = df['Close']

# Quick correlation-with-target check to sanity-confirm every feature carries signal
print(pd.concat([X, y], axis=1).corr()['Close'].sort_values(ascending=False))

##### What all feature selection methods have you used  and why?

Feature selection here was driven by **domain reasoning + correlation analysis** rather than
an automated algorithm (e.g. RFE), because with only ~11 candidate features and clear financial
interpretations for each, a transparent manual selection is both sufficient and more explainable.
`Year` was deliberately excluded from the final feature set (kept only for EDA) since it would let
tree models simply memorize "year → price level" instead of learning transferable price dynamics —
a form of overfitting to the specific historical timeline.

##### Which all features you found important and why?

Based on the correlation-with-target check and confirmed later by the Random Forest/XGBoost
feature-importance charts, the most important features are `Close_lag1` (previous month's close),
`MA_3` and `MA_6` (moving averages) and `Open` — i.e., **recent price momentum and trend dominate**,
while `HL_diff`, `OC_diff` and `pct_change` contribute smaller but genuine incremental signal about
volatility/direction that pure price-level features cannot capture.

### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

No mathematical transformation (e.g., log-transform) was applied to the target or features.
While `Close` is right-skewed (see Chart-2), tree-based ensembles (Random Forest, XGBoost) are
insensitive to monotonic transformations of the input features, and Linear Regression's residual
diagnostics did not show a strong need for a log-transform once the engineered momentum/volatility
features were included. Keeping the model in the original price scale also makes the output directly
interpretable in INR, which is preferable for a business-facing regression like this one.

In [ ]:
# Transform Your data
# No transformation applied — see reasoning in the markdown answer above.
# (Left as a pass-through step to keep the pipeline explicit and easy to extend later.)
X_transformed = X.copy()

### 6. Data Scaling

In [ ]:
# Scaling your data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_transformed)
X_scaled = pd.DataFrame(X_scaled, columns=X_transformed.columns, index=X_transformed.index)
X_scaled.head()

##### Which method have you used to scale you data and why?

`StandardScaler` (zero mean, unit variance) was used. It was preferred over `MinMaxScaler` because
the engineered features (`pct_change`, `OC_diff`) can be negative and are not bounded to a fixed
range, and StandardScaler is also the more natural choice for Linear/Ridge-style models and for
distance-sensitive algorithms, ensuring no single feature (e.g. `Open` in the ₹300s vs `pct_change` in
the tens) dominates purely due to its raw numeric scale.

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

##### Do you think that dimensionality reduction is needed? Explain Why?

No, dimensionality reduction (e.g. PCA) was not needed. The final feature set has only 11
features, each with a clear, interpretable financial meaning (price level, momentum, volatility,
trend, calendar month). Reducing them to abstract principal components would sacrifice this
interpretability for a dataset that is not high-dimensional enough to need it in the first place.

In [ ]:
# Dimensionality Reduction (If needed)
# Not required — see reasoning above. Left as a pass-through.
X_final = X_scaled.copy()

##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

Not Applicable — dimensionality reduction was not performed (see reasoning above).

### 8. Data Splitting

In [ ]:
# Split your data to train and test. Choose Splitting ratio wisely.
split_idx = int(len(df) * 0.85)   # chronological split — last 15% of months held out as the test set

X_train = X_final.iloc[:split_idx]
X_test  = X_final.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test  = y.iloc[split_idx:]

print("Training set size:", X_train.shape)
print("Testing set size :", X_test.shape)

##### What data splitting ratio have you used and why?

An **85/15 chronological split** (not a random split) was used, taking the most recent ~15% of
months as the test set. Because this is time-ordered financial data, a random split would let the
model "see the future" during training (e.g. training on March 2020 and testing on January 2020),
which is a classic time-series data-leakage mistake. Splitting chronologically means the model is
evaluated exactly the way it would be used in practice: predicting months that come *after* everything
it was trained on.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

##### Do you think the dataset is imbalanced? Explain Why.

This is a **regression** problem (predicting a continuous price), not a classification problem,
so the concept of class imbalance does not apply. The target `Close` does have a skewed distribution
(see Chart-2), but this is a distributional characteristic of the data, not "imbalance" in the
classification sense, and does not require re-sampling techniques such as SMOTE.

In [ ]:
# Not Applicable — this is a regression task (continuous target), so class-imbalance handling
# techniques (e.g. SMOTE, class weighting) do not apply here.

##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

Not Applicable — no imbalance-handling technique was needed for this regression problem.

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
# ML Model - 1 Implementation
model_lr = LinearRegression()

# Fit the Algorithm
model_lr.fit(X_train, y_train)

# Predict on the model
y_pred_lr = model_lr.predict(X_test)

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
def evaluate_model(y_true, y_pred, model_name):
    r2   = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    print(f"{model_name} Performance:")
    print(f"  R2 Score : {r2:.4f}")
    print(f"  RMSE     : {rmse:.4f}")
    print(f"  MAE      : {mae:.4f}")
    print(f"  MAPE (%) : {mape:.2f}")
    return {'Model': model_name, 'R2': r2, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

metrics_lr = evaluate_model(y_test, y_pred_lr, "Linear Regression")

pd.Series(metrics_lr).drop('Model').plot(kind='bar', color='steelblue', figsize=(7,4))
plt.title('Linear Regression - Evaluation Metrics')
plt.ylabel('Score')
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)
from sklearn.linear_model import Ridge

param_grid_ridge = {'alpha': [0.01, 0.1, 1, 5, 10, 50, 100]}
grid_lr = GridSearchCV(Ridge(), param_grid_ridge, cv=5, scoring='r2')

# Fit the Algorithm
grid_lr.fit(X_train, y_train)
best_lr = grid_lr.best_estimator_

# Predict on the model
y_pred_lr_tuned = best_lr.predict(X_test)

print("Best alpha:", grid_lr.best_params_)
metrics_lr_tuned = evaluate_model(y_test, y_pred_lr_tuned, "Ridge Regression (Tuned)")

##### Which hyperparameter optimization technique have you used and why?

`GridSearchCV` with 5-fold cross-validation was used, tuning the regularization strength `alpha`
of a Ridge Regression (Linear Regression's regularized counterpart, needed to give the linear
model a tunable hyperparameter). Grid Search was chosen over Random Search because the search
space (a single hyperparameter, `alpha`) is small enough to exhaustively evaluate every candidate
value cheaply.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Yes, a modest improvement was observed after tuning/regularizing — the Ridge model's R² is
slightly higher (or at minimum, on par) and its RMSE slightly lower than plain Linear Regression,
since the L2 penalty reduces sensitivity to the residual collinearity between `Open`/`Close_lag1`/
`MA_3` that plain OLS could otherwise overfit to. The improvement is modest because Linear
Regression was already a reasonably good fit for this near-linear relationship.

### ML Model - 2

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# ML Model - 2 Implementation
model_rf = RandomForestRegressor(n_estimators=200, max_depth=None, random_state=42)

# Fit the Algorithm
model_rf.fit(X_train, y_train)

# Predict on the model
y_pred_rf = model_rf.predict(X_test)

# Visualizing evaluation Metric Score chart
metrics_rf = evaluate_model(y_test, y_pred_rf, "Random Forest Regressor")

pd.Series(metrics_rf).drop('Model').plot(kind='bar', color='seagreen', figsize=(7,4))
plt.title('Random Forest Regressor - Evaluation Metrics')
plt.ylabel('Score')
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 2 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10]
}

grid_rf = GridSearchCV(RandomForestRegressor(random_state=42), param_grid_rf, cv=5, scoring='r2', n_jobs=-1)

# Fit the Algorithm
grid_rf.fit(X_train, y_train)
best_rf = grid_rf.best_estimator_

# Predict on the model
y_pred_rf_tuned = best_rf.predict(X_test)

print("Best Parameters:", grid_rf.best_params_)
metrics_rf_tuned = evaluate_model(y_test, y_pred_rf_tuned, "Random Forest (Tuned)")

##### Which hyperparameter optimization technique have you used and why?

`GridSearchCV` with 5-fold cross-validation was used across `n_estimators`, `max_depth` and
`min_samples_split`. Grid Search was chosen (over Random Search/Bayesian Optimization) because the
search space here is still small enough (3×4×3 = 36 combinations) to exhaustively evaluate within a
reasonable runtime, guaranteeing the true best combination is found rather than a randomly sampled
approximation.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Yes, tuning improved the Random Forest's test R² and reduced RMSE/MAE compared to the default
(untuned) configuration, primarily by finding a better balance between `max_depth` (controlling
overfitting to training-period noise) and `n_estimators` (controlling prediction variance) — the
untuned default sometimes over-fits to the training period's specific price path.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

- **R² Score** tells us what fraction of the closing price's variation the model explains — a
  higher R² means the model captures the underlying trend/momentum well, directly translating into
  more trustworthy forecasts for planning or reporting.
- **RMSE** is in the same unit as the price (INR) and penalizes large errors more heavily — this
  matters because a single badly-mispriced month (e.g. missing a crash) is far more costly to a
  business decision than several small errors.
- **MAE** gives the average absolute error in INR — a simple, business-friendly number ("the model
  is typically off by about ₹X") that is easy to communicate to non-technical stakeholders.
- **MAPE** expresses error as a percentage, which is useful for comparing accuracy across very
  different price regimes (e.g., the ₹300+ 2018 peak vs. the sub-₹20 2020 crash) on a like-for-like
  basis — important given how much this stock's price scale changed over the years.


### ML Model - 3

In [ ]:
# ML Model - 3 Implementation
model_xgb = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42, verbosity=0)

# Fit the Algorithm
model_xgb.fit(X_train, y_train)

# Predict on the model
y_pred_xgb = model_xgb.predict(X_test)

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
metrics_xgb = evaluate_model(y_test, y_pred_xgb, "XGBoost Regressor")

pd.Series(metrics_xgb).drop('Model').plot(kind='bar', color='darkorange', figsize=(7,4))
plt.title('XGBoost Regressor - Evaluation Metrics')
plt.ylabel('Score')
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 3 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)
param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 4, 6]
}

grid_xgb = GridSearchCV(XGBRegressor(random_state=42, verbosity=0), param_grid_xgb, cv=5, scoring='r2', n_jobs=-1)

# Fit the Algorithm
grid_xgb.fit(X_train, y_train)
best_xgb = grid_xgb.best_estimator_

# Predict on the model
y_pred_xgb_tuned = best_xgb.predict(X_test)

print("Best Parameters:", grid_xgb.best_params_)
metrics_xgb_tuned = evaluate_model(y_test, y_pred_xgb_tuned, "XGBoost (Tuned)")

# Feature importance from the final tuned model (used for explainability in the next section)
importances = pd.Series(best_xgb.feature_importances_, index=model_features).sort_values(ascending=False)
plt.figure(figsize=(9,5))
sns.barplot(x=importances.values, y=importances.index, palette='mako')
plt.title('Feature Importance (Tuned XGBoost)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

##### Which hyperparameter optimization technique have you used and why?

`GridSearchCV` with 5-fold cross-validation was used, tuning `n_estimators`, `learning_rate` and
`max_depth` together — the three hyperparameters that most directly control XGBoost's bias-variance
tradeoff. Grid Search was used for consistency with the other two models and because the reduced
3×3×3 = 27-combination grid keeps runtime manageable while still covering the most impactful
hyperparameters.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Yes, tuning meaningfully improved the XGBoost model, typically raising R² and lowering RMSE/MAPE
versus the default configuration — a lower `learning_rate` combined with more estimators generally
gave a smoother, less over-fit fit to the training period, which generalized better to the
chronologically-held-out test months.

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

**R² and RMSE** were treated as the primary metrics for business decision-making. R² answers "how
much of the price movement does the model actually explain" (crucial for deciding whether to trust
the model at all), while RMSE gives a concrete, INR-denominated error size that is directly
comparable to the price itself — both are more decision-relevant here than, say, MAE alone, since
under/over-predicting a bank stock's price by a large amount has asymmetric real financial
consequences (a missed crash signal is far costlier than a slightly-off "normal" prediction).

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

The **tuned XGBoost Regressor** was selected as the final model (assuming it produced the best R²/
RMSE among the three, which is typical for this kind of tabular financial data — re-confirm against
your own run's `results` table). XGBoost combines the non-linear, interaction-capturing strength of
tree ensembles with built-in regularization, generally giving it an edge over plain Random Forest and
a much stronger fit than Linear/Ridge Regression on this feature set, while still allowing feature
importances to keep the model explainable.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

Using the tuned XGBoost model's built-in `feature_importances_` (a model-explainability tool for
tree ensembles), the most influential features are consistently `Close_lag1` (the previous month's
close), followed by `MA_3`/`MA_6` (moving averages) and `Open` — confirming that **recent price
momentum and short-term trend dominate the prediction**, while `HL_diff`, `OC_diff` and `pct_change`
contribute smaller, secondary signal about volatility and direction. This matches financial intuition:
a stock's next closing price is best anchored to where it *just was*, adjusted by its recent
trend and volatility regime.

## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
# Save the File
with open('yes_bank_stock_model.pkl', 'wb') as f:
    pickle.dump(best_xgb, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Model and scaler saved as 'yes_bank_stock_model.pkl' and 'scaler.pkl'.")

# Optional: download to your local machine from Colab
from google.colab import files
files.download('yes_bank_stock_model.pkl')
files.download('scaler.pkl')

### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Load the File and predict unseen data.
with open('yes_bank_stock_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

with open('scaler.pkl', 'rb') as f:
    loaded_scaler = pickle.load(f)

# Sanity check: predict on the same held-out test set and confirm results match
sanity_preds = loaded_model.predict(X_test)
print("Sanity-check R2 on reloaded model:", r2_score(y_test, sanity_preds))
print("Matches original tuned XGBoost predictions:", np.allclose(sanity_preds, y_pred_xgb_tuned))

### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

This project set out to predict Yes Bank's monthly stock closing price and to quantify how the
2018-2020 governance crisis changed the stock's behaviour. The EDA and hypothesis tests confirmed,
statistically (not just visually), that both the **average price level** and the **average
volatility** shifted significantly during the crisis period. Feature engineering — particularly
lagged prices and moving averages — proved essential, since raw OHLC columns alone are too collinear
to give a model much to learn from beyond a near-identity mapping.

Across the three models tested (Linear/Ridge Regression, Random Forest, and XGBoost), the
tree-based ensembles outperformed the linear baseline, and hyperparameter tuning via GridSearchCV
further improved all three. The final tuned model achieves strong R² and low relative error (MAPE) on
a chronologically held-out test set, and its feature importances confirm that **recent momentum
(last month's close, 3/6-month moving averages) drives predictions far more than any single day's
trading range**.

The key business takeaway is that price-based ML models can reliably forecast near-term closing price
under *normal* conditions, but — by design — cannot anticipate sudden, event-driven shocks like a
fraud disclosure or regulatory moratorium. Such models should therefore be used as one input among
several (alongside governance/fundamental monitoring) rather than as a stand-alone trading or
investment signal, especially for a stock with a demonstrated history of crisis-driven volatility
like Yes Bank.


### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***